# Qwen3.5-0.8B × TinyCeNN Memory Fusion — Sequential Acceptance

This notebook ports the TinyCeNN Memory Fusion sequential experiment to `Qwen/Qwen3.5-0.8B`. Qwen3.5 already uses a hybrid layout, so this experiment leaves all native Gated DeltaNet linear-attention layers unchanged and sequentially tries to replace only its six full-attention anchors: **3, 7, 11, 15, 19, 23**.

Each replacement must pass NMSE, cosine, incremental ΔNLL and cumulative ΔNLL before the next anchor is touched. Failed rounds are saved and resumable. The notebook uses the text-only `Qwen3_5ForCausalLM` loader so the vision tower is not loaded for this experiment.

After each run it also compares the original and accepted-only model on prompts and publishes the accepted adapter plus resumable in-progress checkpoint to Hugging Face.


In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
assert subprocess.run(['nvidia-smi'], check=False).returncode == 0, 'Enable a GPU runtime in Colab.'
REPO = Path('/content/TinyCeNN-LM')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.17.0','datasets','huggingface_hub','accelerate','safetensors','pytest'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'], check=True)
for p in (str(REPO), str(REPO/'src')):
    if p not in sys.path: sys.path.insert(0,p)
os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO), str(REPO/'src')])
print('Repository:', REPO)
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())


In [ ]:
from google.colab import drive
from huggingface_hub import HfApi
from transformers import AutoConfig
drive.mount('/content/drive')
BASE_MODEL = 'Qwen/Qwen3.5-0.8B'
MODEL_REVISION = HfApi().model_info(BASE_MODEL).sha
FEATURE_DIM = 32
MEMORY_RANK = 64
CONTEXT = 128
PROBE_CONTEXT = 128
SEED = 73
MAX_ROUNDS_PER_RUN = 4
RESET_PROGRESS = False
HF_MODEL_REPO = 'vtava/Qwen3.5-0.8B-MemoryFusion'
HF_PRIVATE = False
PUBLISH_TO_HF = True
OUTPUT_DIR = Path('/content/drive/MyDrive/TinyCeNN-LM/qwen35-0.8b-memory-fusion-sequential-r64')
if RESET_PROGRESS and OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg = AutoConfig.from_pretrained(BASE_MODEL, revision=MODEL_REVISION).get_text_config(decoder=True)
full_layers = [i for i,k in enumerate(cfg.layer_types) if k == 'full_attention']
linear_layers = [i for i,k in enumerate(cfg.layer_types) if k == 'linear_attention']
print({'model':BASE_MODEL,'revision':MODEL_REVISION,'layers':cfg.num_hidden_layers,'full_attention_layers':full_layers,'native_linear_attention_layers':linear_layers,'hidden_size':cfg.hidden_size,'heads':cfg.num_attention_heads,'kv_heads':cfg.num_key_value_heads,'head_dim':cfg.head_dim,'max_context':cfg.max_position_embeddings,'output':str(OUTPUT_DIR),'hf_repo':HF_MODEL_REPO})
assert full_layers == [3,7,11,15,19,23], full_layers


## Hugging Face authentication
Add a Hugging Face **write** token to Colab Secrets as `HF_TOKEN`. It is used for mandatory live backup and for the final model repository upload.


In [ ]:
from huggingface_hub import HfApi, login, notebook_login
from google.colab import userdata
try: token = userdata.get('HF_TOKEN')
except Exception: token = None
if token:
    os.environ['HF_TOKEN'] = token
    login(token=token, add_to_git_credential=False)
else:
    notebook_login()
print('✅ HF user', HfApi().whoami().get('name'))


In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES='', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
env['TINYCENN_PARENT_BACKUP_ACTIVE'] = '1'
r = subprocess.run([sys.executable,'-m','pytest','-q','tests/test_qwen3_5_memory_fusion.py'], cwd=REPO, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError(f'preflight failed: {r.returncode}')
print('✅ Qwen3.5 preflight passed')


## Train / resume
The trainer is launched directly, not through `bash | tee`, so TinyCeNN's mandatory Hugging Face live-backup wrapper can recognize and protect the training run.


In [ ]:
cmd = [sys.executable,'-u',str(REPO/'scripts'/'train_qwen35_memory_fusion_sequential.py'),'--base-model',BASE_MODEL,'--model-revision',MODEL_REVISION,'--output-dir',str(OUTPUT_DIR),'--feature-dim',str(FEATURE_DIM),'--memory-rank',str(MEMORY_RANK),'--context-length',str(CONTEXT),'--probe-context',str(PROBE_CONTEXT),'--seed',str(SEED),'--min-layer-steps','50','--max-layer-steps','300','--check-every','25','--layer-lr','0.0002','--teacher-alpha-start','0.9','--teacher-alpha-end','0.0','--accept-nmse','0.2','--accept-cosine','0.9','--accept-incremental-delta-nll','0.015','--accept-cumulative-delta-nll','0.05','--max-runtime-minutes','240','--resume','--strict-acceptance']
print(' '.join(cmd), flush=True)
run_env = dict(os.environ)
run_env['SEQUENTIAL_MAX_ROUNDS_PER_RUN'] = str(MAX_ROUNDS_PER_RUN)
result = subprocess.run(cmd, cwd=REPO, env=run_env)
if result.returncode: raise RuntimeError(f'trainer failed with exit code {result.returncode}')
print('✅ trainer returned normally; current_layer_needs_more_training is a resumable scientific status, not a crash')


In [ ]:
import json
for name in ('sequential_run_status.json','sequential_progress.json','sequential_in_progress.json','sequential_training_report.json'):
    p = OUTPUT_DIR/name
    if p.exists():
        print('\n###', name)
        print(p.read_text())


## Compare original Qwen3.5 with accepted Memory Fusion layers
Only accepted layers are loaded. A current unaccepted layer is never used in the inference comparison.


In [ ]:
import torch, json
from transformers import AutoTokenizer, Qwen3_5ForCausalLM
from tinycenn_lm.qwen3_5_memory_fusion import Qwen35MemoryFusionConfig, replace_attention_layers, load_selected_attention_state, structural_summary
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE.type == 'cuda' else torch.float32)
progress_path = OUTPUT_DIR/'sequential_progress.pt'
payload = torch.load(progress_path, map_location='cpu', weights_only=False) if progress_path.exists() else None
accepted = [int(x) for x in payload.get('accepted_layers',[])] if payload else []
mf_cfg = Qwen35MemoryFusionConfig.from_dict(payload['config']) if payload else Qwen35MemoryFusionConfig(feature_dim=FEATURE_DIM,memory_rank=MEMORY_RANK)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, revision=MODEL_REVISION)
original = Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,dtype=DTYPE,attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
adapted = Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,dtype=DTYPE,attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
if accepted:
    replace_attention_layers(adapted,mf_cfg,accepted)
    load_selected_attention_state(adapted,payload['attention_state'],accepted)
adapted.config.use_cache = False
print('Accepted layers:', accepted)
print(json.dumps(structural_summary(adapted), indent=2))
@torch.no_grad()
def gen(model,prompt):
    ids = tok(prompt,return_tensors='pt').input_ids.to(DEVICE)
    out = model.generate(ids,max_new_tokens=48,do_sample=False,use_cache=False,pad_token_id=tok.eos_token_id)
    return tok.decode(out[0],skip_special_tokens=True)
prompts = ['The capital of Austria is','Explain in one sentence why the sky looks blue.','Write a short Python function that adds two numbers.','The largest planet in the Solar System is','2 + 2 =']
examples = []
for i,prompt in enumerate(prompts,1):
    a,b = gen(original,prompt),gen(adapted,prompt)
    examples.append({'prompt':prompt,'original':a,'memory_fusion':b})
    print('\n'+'='*100)
    print('PROMPT',i,prompt)
    print('\nORIGINAL QWEN3.5:\n',a)
    print('\nMEMORY FUSION (accepted only):\n',b)
(OUTPUT_DIR/'prompt_examples.json').write_text(json.dumps(examples,indent=2),encoding='utf-8')
print('✅ prompt smoke test complete')


## Publish accepted adapter + resumable training state to Hugging Face
The inference adapter contains **accepted layers only**. If the current layer has not passed the acceptance gate yet, its checkpoint is uploaded under `training/` for resumption and is not presented as an accepted model layer.


In [ ]:
import shutil, json
from huggingface_hub import HfApi
from tinycenn_lm.qwen3_5_memory_fusion import save_adapter
PACKAGE = Path('/content/qwen35-memory-fusion-hf')
if PACKAGE.exists(): shutil.rmtree(PACKAGE)
PACKAGE.mkdir(parents=True)
training_dir = PACKAGE/'training'
training_dir.mkdir()
progress_path = OUTPUT_DIR/'sequential_progress.pt'
payload = torch.load(progress_path,map_location='cpu',weights_only=False) if progress_path.exists() else None
accepted = [int(x) for x in payload.get('accepted_layers',[])] if payload else []
mf_cfg = Qwen35MemoryFusionConfig.from_dict(payload['config']) if payload else Qwen35MemoryFusionConfig(feature_dim=FEATURE_DIM,memory_rank=MEMORY_RANK)
if accepted:
    save_adapter(adapted, PACKAGE, config=mf_cfg, base_model=BASE_MODEL, accepted_layers=accepted, metadata={'base_revision':MODEL_REVISION,'source_repo':'vtavakkoli/TinyCeNN-LM'})
for name in ('sequential_run_status.json','sequential_progress.json','sequential_in_progress.json','sequential_training_report.json','prompt_examples.json'):
    src = OUTPUT_DIR/name
    if src.exists(): shutil.copy2(src, PACKAGE/name)
for name in ('sequential_in_progress.pt','sequential_progress.pt','qwen35_memory_fusion_full_state.pt'):
    src = OUTPUT_DIR/name
    if src.exists(): shutil.copy2(src, training_dir/name)
reports = []
for source_name in ('sequential_progress.json','sequential_in_progress.json'):
    src = OUTPUT_DIR/source_name
    if src.exists():
        reports = json.loads(src.read_text()).get('layer_reports', reports)
accepted_reports = [r for r in reports if r.get('accepted')]
rows = ['| Layer | NMSE | Cosine | Incremental ΔNLL | Cumulative ΔNLL |','|---:|---:|---:|---:|---:|']
for r in accepted_reports:
    rows.append(f"| {r['layer']} | {r['nmse']:.5f} | {r['cosine']:.5f} | {r['incremental_delta_nll']:+.5f} | {r['cumulative_delta_nll']:+.5f} |")
table = '\n'.join(rows) if accepted_reports else '_No full-attention anchor has passed the acceptance gate yet._'
card = f'''---
library_name: transformers
base_model: {BASE_MODEL}
license: apache-2.0
tags:
- qwen3.5
- tinycenn
- memory-fusion
- recurrent-attention
---
# Qwen3.5-0.8B × TinyCeNN Memory Fusion

Research adapter for `{BASE_MODEL}`. Qwen3.5's native Gated DeltaNet linear-attention layers remain unchanged; TinyCeNN Memory Fusion is trained only for the original full-attention anchors.

**Accepted anchors:** `{accepted}`  
**Target anchors:** `[3, 7, 11, 15, 19, 23]`  
**Base revision:** `{MODEL_REVISION}`  
**Memory Fusion:** feature_dim={FEATURE_DIM}, memory_rank={MEMORY_RANK}

## Acceptance results
{table}

## Files
- `qwen35_memory_fusion.pt`: accepted inference adapter, present after at least one accepted anchor.
- `qwen35_memory_fusion_config.json`: adapter architecture and accepted layers.
- `training/sequential_in_progress.pt`: resumable current layer; it may be **unaccepted** and is not an inference release.
- `prompt_examples.json`: deterministic original-vs-accepted-only smoke tests.

## Scope
This is an experimental attention-replacement adapter, not a claim that Memory Fusion beats the original Qwen3.5 model. The acceptance gates are NMSE ≤ 0.20, cosine ≥ 0.90, incremental ΔNLL ≤ 0.015, and cumulative ΔNLL ≤ 0.05.
'''
(PACKAGE/'README.md').write_text(card,encoding='utf-8')
api = HfApi()
if PUBLISH_TO_HF:
    api.create_repo(HF_MODEL_REPO,repo_type='model',private=HF_PRIVATE,exist_ok=True)
    api.upload_folder(repo_id=HF_MODEL_REPO,repo_type='model',folder_path=str(PACKAGE),commit_message=f'Update Qwen3.5 Memory Fusion accepted={accepted}')
    print('✅ Published:', f'https://huggingface.co/{HF_MODEL_REPO}')
else:
    print('Publication disabled; package ready at', PACKAGE)
